# Public world quality audit

## tl;dr

The empirical count prior loaded **1,794 worlds** without checking the public review ledger. A strict S-tier, independent protocol settlement and reconciliation rule admits **1,284**; **510** are excluded. Of **1,547** protocol-valid labels, **263** have a conflicting item-value sum.

The fixed 15-decision development benchmark keeps **all 15 labels**. Four have conflicting postgame identities; that prevents using their item assignments as a true-world oracle, and excludes those assignments from the empirical prior. It does not establish that their independently valid protocol labels are wrong. This audit does not train, rerun predictions, or select weights.

## Context & Methods

Scoped data-quality companion: session grain is `session_id`; item grain is `(session_id, treasure_id)`; one empirical prior world is one session. Inputs are existing public CSV files and the frozen development receipt.

### Key Assumptions

- The protocol-valid positive settlement is the authority for an event-only target label.
- Item identities require independent review and exact reconciliation before becoming prior samples.
- S-tier count worlds require unique keys, consistent catalog metadata, positive protocol settlement, exact item count and value reconciliation, all review/value gates, independent identity/geometry, and consistent metadata.
- Data conflicts diagnose eligibility; they are not a reason to delete poor benchmark results.
- The frozen generator states 1,556 training sessions but has no joinable training IDs, so this audit cannot certify clean generator training and performs no refit.

In [1]:
from pathlib import Path
import importlib.util
import json
from IPython.display import Markdown, display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/audit_public_world_quality.py").is_file())
source = root / "scripts/audit_public_world_quality.py"
spec = importlib.util.spec_from_file_location("public_quality_audit", source)
audit_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit_module)
report = audit_module.audit(root / "data/v1", root / "results/assistant-conditioning-development-20261019-stable-2s-sanitized.json")

def table(rows, keys):
    lines = ["| " + " | ".join(keys) + " |", "| " + " | ".join("---" for _ in keys) + " |"]
    lines += ["| " + " | ".join(str(row.get(key, "")) for key in keys) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

## Data

Hashes identify the precise public inputs and audit implementation. Paths are repository-relative; no local private path or raw session identifier is exposed. There are 1,816 session rows and 58,068 item rows. Twenty-two sessions have no item world; zero orphan item worlds or duplicate session/item keys were found.

In [2]:
table([{"source": source, "sha256": digest} for source, digest in report["provenance_sha256"].items()], ["source", "sha256"])
table([{"check": key, "count": value} for key, value in report["denominators"].items()], ["check", "count"])

| source | sha256 |
| --- | --- |
| data/v1/core/sessions.csv | 6d1b2f6346335a8e688718d6c3e2cf58941045b65f6afcb5469ebd51416d163c |
| data/v1/core/treasures.csv | ef4df90dcae2d1778c3f2f4ca32103b6262e453a5fa09a66606558439555edac |
| scripts/audit_public_world_quality.py | 96c673ce4a70bb6b4d914c46b2ecdaf481941927867469b9ee8c5a70ef0f909d |
| apps/ephemeral-assistant/src/auction_moment_assistant/prior_quality.py | ae94f3413d5b236846977ef0c80660aa248919f54dd8e50c562f2e4aeddc40d7 |
| models/latest-model-v6-v2.0.0-beta.1/world-model-v2-S.json | 615b219748c1d9920bd3614ee7b83e733eb42c34914c203e90d9553fd6af227c |
| results/assistant-conditioning-development-20261019-stable-2s-sanitized.json | 0c325bd5c194476a3397fa00713ef70150b3d4b01f2aa7eabb8c5c64434d1138 |

| check | count |
| --- | --- |
| session_rows | 1816 |
| unique_session_ids | 1816 |
| treasure_rows | 58068 |
| treasure_worlds | 1794 |
| sessions_without_treasure_world | 22 |
| orphan_treasure_worlds | 0 |
| duplicate_session_ids | 0 |
| duplicate_treasure_keys | 0 |
| conflicting_catalog_ids | 0 |
| sum_mismatch_worlds | 302 |
| protocol_label_eligible_worlds | 1547 |
| protocol_label_eligible_but_sum_mismatch_worlds | 263 |
| prior_eligible_worlds | 1284 |
| prior_excluded_worlds | 510 |
| prior_eligible_without_protocol_requirement | 1489 |
| metadata_contradiction_worlds | 2 |

## Results

The nested rule uses one denominator at a time. Failure counts from separate gates can overlap and must not be added. All item worlds are S-tier. Unknown protocol status is excluded rather than backfilled from a matching item sum.

In [3]:
table(report["prior_rule"]["nested_counts"], ["stage", "input_worlds", "excluded_at_stage", "remaining_worlds"])
table(report["by_date"], ["date", "worlds", "sum_mismatch_worlds", "prior_eligible_worlds", "protocol_label_eligible_worlds"])

| stage | input_worlds | excluded_at_stage | remaining_worlds |
| --- | --- | --- | --- |
| unique_join_and_grain | 1794 | 0 | 1794 |
| s_tier | 1794 | 0 | 1794 |
| catalog_valid_and_consistent | 1794 | 0 | 1794 |
| positive_protocol_settlement | 1794 | 247 | 1547 |
| catalog_and_count_reconciled | 1547 | 263 | 1284 |
| review_and_value_flags_confirmed | 1284 | 0 | 1284 |
| identity_and_geometry_independent | 1284 | 0 | 1284 |
| reconciliation_metadata_consistent | 1284 | 0 | 1284 |

| date | worlds | sum_mismatch_worlds | prior_eligible_worlds | protocol_label_eligible_worlds |
| --- | --- | --- | --- | --- |
| 2026-07-27 | 114 | 15 | 0 | 0 |
| 2026-07-28 | 305 | 49 | 173 | 204 |
| 2026-07-29 | 293 | 53 | 240 | 293 |
| 2026-07-30 | 152 | 32 | 120 | 152 |
| 2026-07-31 | 445 | 70 | 349 | 413 |
| 2026-08-01 | 287 | 45 | 242 | 287 |
| 2026-08-02 | 198 | 38 | 160 | 198 |

### Reconciliation flags and frozen benchmark

Two worlds have matching item sums but conflicting postgame gate metadata; they are already excluded by the strict review rule. The four frozen benchmark identity conflicts are shown below for diagnosis; all 15 event-only labels remain in the benchmark.

In [4]:
table([{"session_id": row["session_id"], "difference": row["value_difference"], "conflicting_fields": ", ".join(row["metadata_contradictions"])} for row in report["metadata_contradictions"]], ["session_id", "difference", "conflicting_fields"])
selected = report["evaluation_decisions"]
assert len(selected) == 15 and all(row["event_only_label_kept"] for row in selected)
assert all(row["protocol_label_eligible"] for row in selected)
table([row for row in selected if not row["catalog_sum_matches_label"]], ["decision_key", "catalog_value_sum", "settlement_label", "value_difference", "event_only_label_kept"])
assert report["denominators"]["prior_eligible_worlds"] + report["denominators"]["prior_excluded_worlds"] == report["denominators"]["treasure_worlds"]
print("Audit assertions passed. All 15 fixed labels retained; no inference or fitting performed.")

| session_id | difference | conflicting_fields |
| --- | --- | --- |
| session_e38d73e6a21b418d | 0 | postgame_value_gate_matches, postgame_value_gate_status |
| session_e783af6ffa579e48 | 0 | postgame_value_gate_matches, postgame_value_gate_status |

| decision_key | catalog_value_sum | settlement_label | value_difference | event_only_label_kept |
| --- | --- | --- | --- | --- |
| session_e6ed9beefcca5d67:R1 | 6629265 | 6479014 | 150251 | True |
| session_bd7cab1ac9797c17:R3 | 7754225 | 4711113 | 3043112 | True |
| session_06211379454e621d:R1 | 5389053 | 5383382 | 5671 | True |
| session_0be69e1de6bdfd2b:R1 | 6780340 | 5496690 | 1283650 | True |

Audit assertions passed. All 15 fixed labels retained; no inference or fitting performed.


## Takeaways

- Confirmed high-severity defect: importing unreconciled item assignments as empirical prior mass can distort identity/value probabilities. Filter prior eligibility and preserve explicit exclusion receipts.
- Confirmed high-severity evaluation limitation: postgame identities are not a valid true-world oracle for four of the 15 fixed decisions. Retain independently valid settlement labels for event-only scoring.
- Protocol status is missing in 215 item worlds and false in 32; the strict gate excludes them without inventing validation.
- Remaining uncertainty: frozen v2 generator training membership is not joinable. Public conflicts cannot prove which fitted generator parameters were affected. No claim of clean training or improved accuracy follows from this audit.

Rerun the saved notebook from the repository root, or run `python scripts/audit_public_world_quality.py --evaluation-receipt results/assistant-conditioning-development-20261019-stable-2s-sanitized.json`. The JSON receipt contains the complete world-by-world reasons and relative source hashes.

Saved public receipt: `results/public-world-quality-audit-20261019.json`. The runtime prior loader is independently checked against this qualification population by `apps/ephemeral-assistant/tests/test_prior_quality.py`.